In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import functools
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import beta as beta_dist
from tqdm.auto import tqdm

from eval import load, passat, typo, util

FIGURE_DIR = Path("../outputs/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 13,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})


In [ ]:
MODEL_NAME = "allenai/OLMo-2-1124-7B-Instruct"
DATASET_CONFIGS = {
    "humaneval": {"dataset_size": 164, "default_numvariants": 51},
    "gsm8k": {"dataset_size": 1000, "default_numvariants": 30},
    "gsm8k_python": {"dataset_size": 1000, "default_numvariants": 11},
    "mmlu": {"dataset_size": 1000, "default_numvariants": 30},
}
DATASETS = ["humaneval", "gsm8k", "gsm8k_python", "mmlu"]
VARIANTS = ["retok", "temperature", "typo"]
DATASET_LABELS = {
    "humaneval": "HumanEval",
    "gsm8k": "GSM8K",
    "gsm8k_python": "GSM8K Python",
    "mmlu": "MMLU",
}
DATASET_COLORS = {
    "humaneval": "blue",
    "gsm8k": "orange",
    "gsm8k_python": "red",
    "mmlu": "green",
}
VARIANT_LINESTYLES = {
    "temperature": "-",
    "retok": "--",
    "typo": ":",
}
VARIANT_COLORS = {
    "temperature": "seagreen",
    "retok": "firebrick",
    "typo": "royalblue",
}
VARIANT_LABELS = {
    "temperature": "pass@k",
    "retok": "pass@retok",
    "typo": "pass@typo",
}
LOW_TAIL_THRESHOLD = 0.1
HIGH_TAIL_THRESHOLD = 0.9

LOADERS = {
    "humaneval": load.load_humaneval,
    "gsm8k": load.load_gsm8k,
    "gsm8k_python": load.load_gsm8k_python,
    "mmlu": load.load_mmlu,
}

def requested_numvariants(dataset: str, variant: str) -> int:
    if dataset == "gsm8k_python" or variant == "typo":
        return 11
    return int(DATASET_CONFIGS[dataset]["default_numvariants"])

def load_variant(dataset: str, variant: str) -> pd.DataFrame:
    config = DATASET_CONFIGS[dataset]
    return LOADERS[dataset](
        model_name=MODEL_NAME,
        dataset_size=int(config["dataset_size"]),
        numvariants=requested_numvariants(dataset, variant),
        variant_type=variant,
    )

data = {
    dataset: {variant: load_variant(dataset, variant) for variant in VARIANTS}
    for dataset in DATASETS
}

tokenizer = util.load_tokenizer(MODEL_NAME)
special_ids = set(tokenizer.all_special_ids)


In [ ]:
def calculate_pass_k(n_total: int, num_correct: int, k: int) -> float:
    incorrect = n_total - num_correct
    if incorrect < k:
        return 1.0
    return 1.0 - comb(incorrect, k) / comb(n_total, k)

def recover_prompt_tokenization(prompt, generation_tokens):
    lower_bound = len(tokenizer.encode(prompt, add_special_tokens=False))
    for j in range(max(1, lower_bound), len(generation_tokens) + 1):
        prefix = generation_tokens[:j]
        decoded_prefix = tokenizer.decode(
            prefix,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        if decoded_prefix == prompt:
            prompt_length = sum(token_id not in special_ids for token_id in prefix)
            return int(prompt_length), int(j)
    raise ValueError(f"Could not recover prompt boundary for prompt prefix {prompt[:80]!r}")

def recover_typo_docstring_prompt_tokenization(prompt, generation_tokens):
    decoded_text = tokenizer.decode(
        list(generation_tokens),
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    docstring_span = typo._find_docstring_span(decoded_text)
    if docstring_span is None:
        raise ValueError(f"Could not recover typo docstring span for prompt prefix {prompt[:80]!r}")
    docstring_start, docstring_end = docstring_span
    prompt_char_boundary = docstring_end + 3
    if prompt_char_boundary < len(decoded_text) and decoded_text[prompt_char_boundary] == "\n":
        prompt_char_boundary += 1
    prompt_length = 0
    prompt_index = 0
    char_cursor = 0
    for raw_index, token_id in enumerate(generation_tokens, start=1):
        if token_id in special_ids:
            continue
        surface = tokenizer.decode([int(token_id)], skip_special_tokens=False, clean_up_tokenization_spaces=False)
        token_start = char_cursor
        token_end = token_start + len(surface)
        char_cursor = token_end
        prompt_length += 1
        if prompt_index == 0 and token_end >= prompt_char_boundary:
            prompt_index = raw_index
            break
    if prompt_index == 0:
        raise ValueError(f"Recovered typo docstring span but not prompt boundary for {prompt[:80]!r}")
    return int(prompt_length), int(prompt_index)

def attach_prompt_lengths(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "prompt_tokenization_length" in df.columns and df["prompt_tokenization_length"].notna().all():
        return df
    if "token_lengths" in df.columns and df["token_lengths"].notna().all():
        df["prompt_tokenization_length"] = df["token_lengths"].map(lambda token_len: token_len[1])
        return df
    lengths = np.zeros(len(df), dtype=int)
    prompt_indices = np.zeros(len(df), dtype=int)
    if df.attrs.get("variant_type") == "temperature":
        canonical_by_task = {}
        for task_id, task_df in tqdm(df.groupby("task_id", sort=False), total=df["task_id"].nunique(), desc="Recovering canonical prompt lengths"):
            canonical_by_task[task_id] = recover_prompt_tokenization(task_df["prompt"].iloc[0], task_df["generation_tokens"].iloc[0])
        for row_index, task_id in enumerate(df["task_id"]):
            lengths[row_index], prompt_indices[row_index] = canonical_by_task[task_id]
    else:
        recovery_fn = recover_typo_docstring_prompt_tokenization if df.attrs.get("variant_type") == "typo" else recover_prompt_tokenization
        for row_index, (prompt, generation_tokens) in tqdm(
            enumerate(df[["prompt", "generation_tokens"]].itertuples(index=False, name=None)),
            total=len(df),
            desc=f"Recovering prompt lengths for {df.attrs.get('dataset')} / {df.attrs.get('variant_type')}",
        ):
            lengths[row_index], prompt_indices[row_index] = recovery_fn(prompt, generation_tokens)
    df["prompt_tokenization_length"] = lengths
    df["prompt_tokenization_index"] = prompt_indices
    return df

def hypergeom_q_prob(n, c, k, q):
    if q < 0 or q > c or (k - q) < 0 or (k - q) > (n - c) or k > n:
        return 0.0
    return comb(c, q) * comb(n - c, k - q) / comb(n, k)

def expected_prompt_length(summary_row, k):
    n = int(summary_row["num_samples"])
    c = int(summary_row["num_correct"])
    mean_correct = float(summary_row["mean_correct_prompt_length"])
    mean_wrong = float(summary_row["mean_wrong_prompt_length"])
    total = 0.0
    for q in range(1, k + 1):
        p_q = hypergeom_q_prob(n, c, k, q)
        if p_q == 0.0:
            continue
        total += p_q * (q * mean_correct + max(k - q, 0) * mean_wrong)
    return total

def summarize_prompt_lengths(df: pd.DataFrame) -> pd.DataFrame:
    def per_task(task_df):
        passed = task_df["passed"].astype(bool)
        return pd.Series({
            "num_samples": len(task_df),
            "num_correct": int(passed.sum()),
            "mean_correct_prompt_length": task_df.loc[passed, "prompt_tokenization_length"].mean(),
            "mean_wrong_prompt_length": task_df.loc[~passed, "prompt_tokenization_length"].mean(),
        })
    return df.groupby("task_id", sort=True).apply(per_task).reset_index()

def empirical_failure_probabilities(df: pd.DataFrame) -> pd.DataFrame:
    dataset = df.attrs.get("dataset")
    variant_type = df.attrs.get("variant_type")
    if dataset == "mmlu" and variant_type == "temperature":
        summary = passat.summarize_task_answer_probabilities(df)
        return summary.assign(q=1.0 - summary["answer_prob"].astype(float))[["task_id", "q"]]
    summary = passat.summarize_task_outcomes(df)
    return summary.assign(q=(summary["num_samples"] - summary["num_correct"]) / summary["num_samples"])[["task_id", "q"]]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for dataset in DATASETS:
    if dataset == "mmlu":
        continue
    for variant in ["retok", "temperature"]:
        x, y, yerr = passat.pass_curve_points(data[dataset][variant], max_k=50)
        ax.fill_between(x, y - yerr, y + yerr, color=DATASET_COLORS[dataset], alpha=0.05)
        ax.plot(x, y, color=DATASET_COLORS[dataset], linestyle=VARIANT_LINESTYLES[variant])
dataset_handles = [ax.plot([], [], color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset])[0] for dataset in DATASETS]
metric_handles = [ax.plot([], [], color="black", linestyle=VARIANT_LINESTYLES[variant], label=VARIANT_LABELS[variant])[0] for variant in ["temperature", "retok"]]
legend_1 = ax.legend(handles=dataset_handles, loc="lower right", frameon=True)
ax.add_artist(legend_1)
ax.legend(handles=metric_handles, loc="lower center", frameon=True)
ax.set_xlim(1, 50)
ax.set_xlabel("k")
ax.set_ylabel("Pass Rate")
ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / "2_passat_retok.svg", bbox_inches="tight")

fig, ax = plt.subplots(figsize=(7, 5))
for dataset in DATASETS:
    for variant in VARIANTS:
        x, y, yerr = passat.pass_curve_points(data[dataset][variant], max_k=50)
        ax.fill_between(x, y - yerr, y + yerr, color=DATASET_COLORS[dataset], alpha=0.05)
        ax.plot(x, y, color=DATASET_COLORS[dataset], linestyle=VARIANT_LINESTYLES[variant])
dataset_handles = [ax.plot([], [], color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset])[0] for dataset in DATASETS]
metric_handles = [ax.plot([], [], color="black", linestyle=VARIANT_LINESTYLES[variant], label=VARIANT_LABELS[variant])[0] for variant in VARIANTS]
legend_1 = ax.legend(handles=dataset_handles, loc="lower right", frameon=True)
ax.add_artist(legend_1)
ax.legend(handles=metric_handles, loc="lower center", frameon=True)
ax.set_xlim(1, 50)
ax.set_xlabel("k")
ax.set_ylabel("Pass Rate")
ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / "2_passat_all.svg", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "2_passat_all.png", bbox_inches="tight")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5), constrained_layout=True)
ax_curve, ax_dist = axes
for variant in ["temperature", "retok", "typo"]:
    df = data["humaneval"][variant]
    x, y, yerr = passat.pass_curve_points(df, max_k=50)
    ax_curve.fill_between(x, y - yerr, y + yerr, color=VARIANT_COLORS[variant], alpha=0.2)
    ax_curve.plot(x, y, label=VARIANT_LABELS[variant], color=VARIANT_COLORS[variant])
    p_failure = 1 - df.groupby("task_id").passed.mean().values
    counts, bins = np.histogram(p_failure, bins=np.linspace(0, 1, 25))
    counts = counts / counts.sum()
    ax_dist.step(bins, np.append(counts, counts[-1]), where="post", label=VARIANT_LABELS[variant], color=VARIANT_COLORS[variant])
    ax_dist.axvline(np.mean(p_failure), color=VARIANT_COLORS[variant], linestyle="--", linewidth=1)
ax_curve.set_xlim(0, 50)
ax_curve.set_xlabel("k")
ax_curve.set_ylabel("Pass Rate")
ax_curve.legend(frameon=False)
ax_dist.set_xlabel(r"$P_{fail}$")
ax_dist.set_ylabel("Fraction of Tasks")
ax_dist.set_xticks(np.linspace(0, 1, 6))
fig.savefig(FIGURE_DIR / "7_passattypo_humaneval.svg", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "7_passattypo_humaneval.png", bbox_inches="tight")


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for dataset in DATASETS:
    if dataset == "mmlu":
        continue
    retok_df = data[dataset]["retok"]
    temp_df = data[dataset]["temperature"]
    q_retok = 1 - retok_df.groupby("task_id").passed.mean().values
    q_temp = 1 - temp_df.groupby("task_id").passed.mean().values
    counts, bins = np.histogram(q_temp - q_retok, bins=np.linspace(-1, 1, 21))
    counts = counts / counts.sum()
    ax.step(bins[:-1], counts, where="pre", label=DATASET_LABELS[dataset], color=DATASET_COLORS[dataset], alpha=0.8)
ax.set_xticks(np.linspace(-1, 1, 6))
ax.set_xlabel(r"$\Delta P = P_{fail}(canon) - P_{fail}(retok)$")
ax.set_ylabel("Density (Normalized)")
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / "2_deltaP_retok.svg", bbox_inches="tight")

fig, ax = plt.subplots(figsize=(5, 5))
for dataset in DATASETS:
    typo_df = data[dataset]["typo"]
    temp_df = data[dataset]["temperature"]
    q_typo = 1 - typo_df.groupby("task_id").passed.mean().values
    if dataset == "mmlu":
        q_temp = 1 - temp_df[temp_df.p == 0].answer_prob.values
    else:
        q_temp = 1 - temp_df.groupby("task_id").passed.mean().values
    counts, bins = np.histogram(q_temp - q_typo, bins=np.linspace(-1, 1, 21))
    counts = counts / counts.sum()
    ax.step(bins[:-1], counts, where="pre", label=DATASET_LABELS[dataset], color=DATASET_COLORS[dataset], alpha=0.8)
ax.set_xticks(np.linspace(-1, 1, 11))
ax.set_xlabel(r"$\Delta P = P(failure|canonical) - P(failure|typo)$")
ax.set_ylabel("Density (Normalized)")
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)
fig.savefig(FIGURE_DIR / "deltaP_typo.svg", bbox_inches="tight")

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for ax, dataset in zip(axes.flatten(), DATASETS):
    retok_df = data[dataset]["retok"]
    temp_df = data[dataset]["temperature"]
    q_retok = 1 - retok_df.groupby("task_id").passed.mean().values
    if dataset == "mmlu":
        q_temp = 1 - temp_df[temp_df.p == 0].answer_prob.values
    else:
        q_temp = 1 - temp_df.groupby("task_id").passed.mean().values
    for q_values, color, label in [(q_retok, "firebrick", "pass@retok"), (q_temp, "royalblue", "pass@k")]:
        counts, bins = np.histogram(q_values, bins=np.linspace(0, 1, 25))
        counts = counts / counts.sum()
        ax.bar(bins[:-1], counts, width=bins[1] - bins[0], alpha=0.5, label=label, color=color, align="edge", edgecolor="black")
    ax.set_title(f"{DATASET_LABELS[dataset]} ({len(q_retok)} problems)")
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(loc="upper left")
for ax in axes[:, 0]:
    ax.set_ylabel("Density (Normalized)")
for ax in axes[1, :]:
    ax.set_xlabel(r"$P_{fail}$")
fig.savefig(FIGURE_DIR / "P_failure.svg", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "P_failure.png", bbox_inches="tight")


In [ ]:
annotated_humaneval = {
    variant: attach_prompt_lengths(data["humaneval"][variant])
    for variant in ["temperature", "retok", "typo"]
}
humaneval_lk_curves = []
for variant, df in annotated_humaneval.items():
    task_summary = summarize_prompt_lengths(df)
    max_k = min(50, int(task_summary["num_samples"].min()))
    ks = np.arange(1, max_k + 1, step=3, dtype=int)
    humaneval_lk_curves.append(pd.DataFrame({
        "k": ks,
        "L(k)": [task_summary.apply(expected_prompt_length, axis=1, k=int(k)).mean() for k in ks],
        "L(k)_std": [task_summary.apply(expected_prompt_length, axis=1, k=int(k)).std() / np.sqrt(int(task_summary["num_samples"].min())) for k in ks],
        "variant": variant,
    }))
humaneval_lk_curves = pd.concat(humaneval_lk_curves, ignore_index=True)

fig, ax = plt.subplots(figsize=(5, 5))
for variant in ["temperature", "retok", "typo"]:
    curve = humaneval_lk_curves[humaneval_lk_curves["variant"] == variant]
    pass_curve = passat.pass_curve_dataframe(annotated_humaneval[variant], ks=curve["k"].tolist())
    ax.plot(pass_curve["pass_rate"], curve["L(k)"] / curve["k"], label=VARIANT_LABELS[variant], marker="o", markersize=5, color=VARIANT_COLORS[variant], linestyle="-")
ax.set_ylabel(r"$\frac{\bar{L}(k)}{k}$", fontsize=20, rotation=0)
ax.set_xlabel("Pass@ rate")
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
fig.savefig(FIGURE_DIR / "compute_l_k_over_k.svg", bbox_inches="tight")

summary_rows = []
for dataset in DATASETS:
    for variant in VARIANTS:
        q_df = empirical_failure_probabilities(data[dataset][variant])
        q_values = q_df["q"].to_numpy(dtype=float)
        alpha, beta = beta_dist.fit(np.clip(q_values, 1e-6, 1 - 1e-6), floc=0.0, fscale=1.0)[:2]
        summary_rows.append({
            "dataset": dataset,
            "variant": variant,
            "empirical_mass_A": float(np.mean(q_values < LOW_TAIL_THRESHOLD)),
            "empirical_mass_B": float(np.mean(q_values > HIGH_TAIL_THRESHOLD)),
            "fit_mass_A": float(beta_dist.cdf(LOW_TAIL_THRESHOLD, alpha, beta)),
            "fit_mass_B": float(1.0 - beta_dist.cdf(HIGH_TAIL_THRESHOLD, alpha, beta)),
        })
summary_df = pd.DataFrame(summary_rows)

curve_data = {(dataset, variant): data[dataset][variant] for dataset in DATASETS for variant in VARIANTS}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
ax_curve, ax_scatter = axes
for dataset in DATASETS:
    color = DATASET_COLORS[dataset]
    for variant in VARIANTS:
        x, y, yerr = passat.pass_curve_points(curve_data[(dataset, variant)], max_k=50)
        ax_curve.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.05)
        ax_curve.plot(x, y, color=color, linestyle=VARIANT_LINESTYLES[variant])
dataset_handles = [ax_curve.plot([], [], color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset])[0] for dataset in DATASETS]
variant_handles = [ax_curve.plot([], [], color="black", linestyle=VARIANT_LINESTYLES[variant], label=VARIANT_LABELS[variant])[0] for variant in VARIANTS]
legend_1 = ax_curve.legend(handles=dataset_handles, title="Dataset", loc="lower right", frameon=True)
ax_curve.add_artist(legend_1)
ax_curve.legend(handles=variant_handles, title="Metric", loc="lower center", frameon=True)
ax_curve.set_xlim(1, 50)
ax_curve.set_xlabel("k", fontsize=15)
ax_curve.set_ylabel("Pass Rate", fontsize=15)
ax_curve.grid(True, alpha=0.3)
variant_markers = {"temperature": "o", "retok": "s", "typo": "^"}
for row in summary_df.itertuples(index=False):
    ax_scatter.scatter(row.empirical_mass_A, row.empirical_mass_B, s=150, edgecolors="black", color=DATASET_COLORS[row.dataset], marker=variant_markers[row.variant], linewidths=1, alpha=0.65)
    if row.variant == "temperature":
        ax_scatter.annotate(row.dataset, (row.empirical_mass_A, row.empirical_mass_B), textcoords="offset points", xytext=(-20, 10), fontsize=12, color=DATASET_COLORS[row.dataset])
scatter_x = np.linspace(0, 1, 100)
ax_scatter.plot(scatter_x, 1 - scatter_x, linestyle="--", color="gray", alpha=0.7)
variant_scatter_handles = [plt.Line2D([0], [0], marker=variant_markers[variant], color="black", linestyle="none", markersize=12, label=variant) for variant in VARIANTS]
ax_scatter.legend(handles=variant_scatter_handles, loc=(0.65, 0.7), frameon=True, title="Sampling Method")
ax_scatter.set_xlim(-0.05, 0.8)
ax_scatter.set_ylim(-0.05, 0.8)
ax_scatter.set_xlabel(r"$\mathbb{E}[q < 0.1]_{q \sim P_{fail}}$", fontsize=15)
ax_scatter.set_ylabel(r"$\mathbb{E}[q > 0.9]_{q \sim P_{fail}}$", fontsize=15)
ax_scatter.grid(alpha=0.3)
fig.savefig(FIGURE_DIR / "olmo2_failure_tail_mass.png", dpi=300, bbox_inches="tight")


In [ ]:
def attach_total_processed_lengths(df: pd.DataFrame) -> pd.DataFrame:
    df = attach_prompt_lengths(df)
    df = df.copy()
    if "total_processed_tokenization_length" in df.columns and df["total_processed_tokenization_length"].notna().all():
        return df
    total_lengths = df["generation_tokens"].map(
        lambda generation_tokens: sum(int(token_id) not in special_ids for token_id in generation_tokens)
    )
    df["total_processed_tokenization_length"] = total_lengths.astype(int)
    df["completion_tokenization_length"] = (
        df["total_processed_tokenization_length"] - df["prompt_tokenization_length"]
    ).astype(int)
    return df

def expected_total_processed_tokens(summary_row, k):
    n = int(summary_row["num_samples"])
    c = int(summary_row["num_correct"])
    mean_correct = float(summary_row["mean_correct_total_processed_length"])
    mean_wrong = float(summary_row["mean_wrong_total_processed_length"])
    total = 0.0
    for q in range(1, k + 1):
        p_q = hypergeom_q_prob(n, c, k, q)
        if p_q == 0.0:
            continue
        total += p_q * (q * mean_correct + max(k - q, 0) * mean_wrong)
    return total

def summarize_total_processed_lengths(df: pd.DataFrame) -> pd.DataFrame:
    def per_task(task_df):
        passed = task_df["passed"].astype(bool)
        return pd.Series({
            "num_samples": len(task_df),
            "num_correct": int(passed.sum()),
            "mean_correct_total_processed_length": task_df.loc[passed, "total_processed_tokenization_length"].mean(),
            "mean_wrong_total_processed_length": task_df.loc[~passed, "total_processed_tokenization_length"].mean(),
        })
    return df.groupby("task_id", sort=True).apply(per_task).reset_index()

annotated_humaneval_total = {
    variant: attach_total_processed_lengths(data["humaneval"][variant])
    for variant in ["temperature", "retok", "typo"]
}
humaneval_total_tk_curves = []
for variant, df in annotated_humaneval_total.items():
    task_summary = summarize_total_processed_lengths(df)
    max_k = min(50, int(task_summary["num_samples"].min()))
    ks = np.arange(1, max_k + 1, step=3, dtype=int)
    humaneval_total_tk_curves.append(pd.DataFrame({
        "k": ks,
        "T(k)": [task_summary.apply(expected_total_processed_tokens, axis=1, k=int(k)).mean() for k in ks],
        "variant": variant,
    }))
humaneval_total_tk_curves = pd.concat(humaneval_total_tk_curves, ignore_index=True)

fig, ax = plt.subplots(figsize=(5.5, 5))
for variant in ["temperature", "retok", "typo"]:
    curve = humaneval_total_tk_curves[humaneval_total_tk_curves["variant"] == variant]
    pass_curve = passat.pass_curve_dataframe(annotated_humaneval_total[variant], ks=curve["k"].tolist())
    ax.plot(
        pass_curve["pass_rate"],
        curve["T(k)"],
        label=VARIANT_LABELS[variant],
        marker="o",
        markersize=5,
        color=VARIANT_COLORS[variant],
        linestyle="-",
    )
ax.set_ylabel(r"$\bar{T}(k)$ (prompt + completion tokens)")
ax.set_xlabel("Pass@ rate")
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
fig.savefig(FIGURE_DIR / "compute_l_k_prompt_plus_completion.svg", bbox_inches="tight")


In [ ]:
def attach_total_processed_lengths(df: pd.DataFrame) -> pd.DataFrame:
    df = attach_prompt_lengths(df)
    df = df.copy()
    if "total_processed_tokenization_length" in df.columns and df["total_processed_tokenization_length"].notna().all():
        return df
    total_lengths = df["generation_tokens"].map(
        lambda generation_tokens: sum(int(token_id) not in special_ids for token_id in generation_tokens)
    )
    df["total_processed_tokenization_length"] = total_lengths.astype(int)
    df["completion_tokenization_length"] = (
        df["total_processed_tokenization_length"] - df["prompt_tokenization_length"]
    ).astype(int)
    return df

def expected_total_processed_tokens(summary_row, k):
    n = int(summary_row["num_samples"])
    c = int(summary_row["num_correct"])
    mean_correct = float(summary_row["mean_correct_total_processed_length"])
    mean_wrong = float(summary_row["mean_wrong_total_processed_length"])
    total = 0.0
    for q in range(1, k + 1):
        p_q = hypergeom_q_prob(n, c, k, q)
        if p_q == 0.0:
            continue
        total += p_q * (q * mean_correct + max(k - q, 0) * mean_wrong)
    return total

def expected_completion_tokens(summary_row, k):
    n = int(summary_row["num_samples"])
    c = int(summary_row["num_correct"])
    mean_correct = float(summary_row["mean_correct_completion_tokenization_length"])
    mean_wrong = float(summary_row["mean_wrong_completion_tokenization_length"])
    total = 0.0
    for q in range(1, k + 1):
        p_q = hypergeom_q_prob(n, c, k, q)
        if p_q == 0.0:
            continue
        total += p_q * (q * mean_correct + max(k - q, 0) * mean_wrong)
    return total

def summarize_total_processed_lengths(df: pd.DataFrame) -> pd.DataFrame:
    def per_task(task_df):
        passed = task_df["passed"].astype(bool)
        return pd.Series({
            "num_samples": len(task_df),
            "num_correct": int(passed.sum()),
            "mean_correct_total_processed_length": task_df.loc[passed, "total_processed_tokenization_length"].mean(),
            "mean_wrong_total_processed_length": task_df.loc[~passed, "total_processed_tokenization_length"].mean(),
            "mean_correct_completion_tokenization_length": task_df.loc[passed, "completion_tokenization_length"].mean(),
            "mean_wrong_completion_tokenization_length": task_df.loc[~passed, "completion_tokenization_length"].mean(),
        })
    return df.groupby("task_id", sort=True).apply(per_task).reset_index()

annotated_humaneval_total = {
    variant: attach_total_processed_lengths(data["humaneval"][variant])
    for variant in ["temperature", "retok", "typo"]
}
humaneval_total_tk_curves = []
for variant, df in annotated_humaneval_total.items():
    task_summary = summarize_total_processed_lengths(df)
    max_k = min(50, int(task_summary["num_samples"].min()))
    ks = np.arange(1, max_k + 1, step=3, dtype=int)
    humaneval_total_tk_curves.append(pd.DataFrame({
        "k": ks,
        "T(k)": [task_summary.apply(expected_total_processed_tokens, axis=1, k=int(k)).mean() for k in ks],
        "O(k)":[task_summary.apply(expected_completion_tokens, axis=1, k=int(k)).mean() for k in ks],
        "variant": variant,
    }))
humaneval_total_tk_curves = pd.concat(humaneval_total_tk_curves, ignore_index=True)

fig, ax = plt.subplots(figsize=(5.5, 5))
for variant in ["temperature", "retok", "typo"]:
    curve = humaneval_total_tk_curves[humaneval_total_tk_curves["variant"] == variant]
    pass_curve = passat.pass_curve_dataframe(annotated_humaneval_total[variant], ks=curve["k"].tolist())
    ax.plot(
        pass_curve["pass_rate"],
        curve["O(k)"],
        label=VARIANT_LABELS[variant],
        marker="o",
        markersize=5,
        color=VARIANT_COLORS[variant],
        linestyle="-",
    )
ax.set_ylabel(r"$\bar{O}(k)$ (completion tokens)")
ax.set_xlabel("Pass@ rate")
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
# fig.savefig(FIGURE_DIR / "compute_l_k_prompt_plus_completion.svg", bbox_inches="tight")


In [ ]:
curve

In [ ]:
humaneval_total_tk_curves[humaneval_total_tk_curves["variant"] == variant]

In [ ]:
annotated_humaneval_total['temperature']

In [ ]:
humaneval_total_tk_curves